# S8 - Procesamiento en streaming con Spark: ventanas, watermarking y semantica de entrega

**Actividad:** construir un pipeline de Spark Structured Streaming sobre `atmos-eventos` (el mismo topic de Kafka que S7 ya deja publicando telemetria de sensores), aplicando agregaciones por ventana de tiempo, watermarking para datos tardios, checkpointing para tolerancia a fallos, deduplicacion acotada, y comparando semantica de entrega y throughput/latencia bajo distintos parametros.


## Objetivo

1. Verificar que Kafka y el productor/consumidor de S7 siguen funcionando, y unir `pyspark` a la red de Kafka.
2. Crear la `SparkSession` con el conector de Kafka.
3. Leer `atmos-eventos` en modo streaming y parsear el JSON con un esquema explicito.
4. Ver los eventos llegar en vivo, por consola, y persistir esa misma lectura en Parquet.
5. Agregar por ventana de tiempo, con y sin watermarking, y comparar el resultado.
6. Agregar por ventana deslizante e inspeccionar el contenido real de un checkpoint.
7. Provocar un dato tardio descartado por el watermark y un duplicado filtrado por `dropDuplicates`.
8. Comparar el costo de dos intervalos de disparo distintos.

## 3.1 Reanudar el entorno `lambda26` y verificar el punto de partida

**Producto del paso:** Kafka, Kafka UI y el simulador de sensores de S7 (`uso-atmos`) corriendo, con eventos reales llegando a `atmos-eventos`; y `pyspark` unido a la red de Kafka, algo que hasta hoy nunca necesito.

Este notebook **no genera sus propios datos**: consume el mismo topic que S7 ya construyo. Antes de continuar, en una terminal (fuera de este notebook):

```bash
cd kafka
docker compose up -d
cd ../uso-atmos
docker compose up -d
docker exec -it lambda26-uso-atmos python /app/consumer_sensores.py
```

Sin `-d`: el consumidor de S7 arranca pegado a esta terminal para que veas en vivo, linea por linea, que los eventos siguen llegando bien formados antes de confiar en la lectura nueva de Spark. En cuanto veas pasar algunas lineas con `"status": "consumed"`, detenlo con `Ctrl+C` — ya cumplio su funcion, no hace falta dejarlo corriendo. Abre una **pestana nueva** de terminal para el productor:

```bash
cd uso-atmos
docker exec -it lambda26-uso-atmos python /app/producer_sensores.py
```

Tambien sin `-d`, pero al reves que el consumidor: el productor **si** debe quedar corriendo toda la sesion — varias celdas de este notebook necesitan eventos **llegando en vivo**, no solo los que ya estan en el topic. Deja esta pestana abierta, visible, sin tocarla (ni `Ctrl+C` ni cerrarla) hasta terminar la sesion, y abre una **tercera pestana** para el resto de los comandos de este paso.

**Error frecuente**: si detienes el productor sin querer (`Ctrl+C` en su pestana, o cerrandola), las celdas que dependen de datos en vivo dejan de recibir mensajes nuevos — no fallan con un error, simplemente no muestran filas nuevas. Como corre sin `-d`, confirmar que sigue vivo es mirar su pestana: debe seguir imprimiendo una linea `"status": "published"` cada pocos segundos. Si se detuvo, vuelve a lanzar el comando de arriba.

**Error frecuente**: el consumidor sigue recibiendo eventos aunque creas que ya no hay ningun productor corriendo, y nunca usaste `-d`. Sin `-it`, `docker exec` no asigna una pseudo-terminal: `Ctrl+C` puede cortar la conexion de tu lado (el prompt vuelve) sin que la senal llegue de verdad al proceso de adentro, que queda corriendo invisible — el mismo sintoma de **zombie**, aunque nunca lo lanzaste detached. Por eso los comandos de arriba ya llevan `-it`. Si de todas formas sospechas que quedo algo vivo (por ejemplo, porque corriste el comando viejo sin `-it`, o con `-d`), confirmalo:

```bash
docker top lambda26-uso-atmos
```

Si aparece mas de un `producer_sensores.py` (o uno que no reconoces), hay un zombie. **No intentes matarlo por PID** con `docker exec lambda26-uso-atmos bash -c "kill <PID>"`: los PID que muestra `docker top` son del *host* (la VM de Docker Desktop), no los del namespace interno del contenedor, asi que casi siempre da `No such process` aunque el proceso este vivo. La forma confiable de limpiar todo es reiniciar el contenedor:

```bash
docker restart lambda26-uso-atmos
```

Esto mata **todos** los procesos de adentro (productor, consumidor, cualquier zombie) y lo deja en su estado base (`sleep infinity`). Vuelve a lanzar consumidor y productor limpios, como al principio de este paso.

`pyspark/compose.yml` corre desde S1, y nunca tuvo que hablar con Kafka: hasta S7, Kafka lo usaban otros contenedores (`uso-atmos`, `ec-eventos-py`), nunca Spark. Hoy cambia: `spark.read.format("kafka")` (mas abajo) corre **dentro** del contenedor `pyspark`, y ese contenedor todavia no esta en la misma red que `lambda26-kafka` — sin unirlo, `kafka:9092` no se resuelve. En vez de tocar `pyspark/compose.yml` (que sirve para todo el curso, no solo para streaming), crea un segundo archivo que solo agrega esa red:

**`pyspark/compose.kafka.yml`:**

```yaml
services:
  pyspark:
    networks:
      - default
      - lambda26-kafka-net

networks:
  lambda26-kafka-net:
    external: true
    name: lambda26-kafka-net
```

Subelo combinando los dos archivos — el segundo solo agrega la red, no reemplaza nada del primero:

```bash
cd pyspark
docker compose -f compose.yml -f compose.kafka.yml up -d
```

**Error frecuente**: la celda de abajo falla con `kafka.errors.NoBrokersAvailable` o, en el log del driver, `java.net.UnknownHostException: kafka`. `pyspark` esta corriendo, pero sin `compose.kafka.yml` — subelo de nuevo con los dos `-f` del comando de arriba.

## 3.2 Crear el notebook y la `SparkSession`, con el conector de Kafka

**Producto del paso:** notebook `08_streaming_estructurado_practica.ipynb` con una `SparkSession` capaz de leer y escribir Kafka.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion8-streaming-estructurado")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark


:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6e6bd585-57e5-49f8-a4ae-cdc9cc7017c1;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

`spark.jars.packages` agrega el conector `spark-sql-kafka-0-10`, que Spark no trae por defecto (a diferencia de la lectura de CSV o Parquet de S2-S4). La primera vez que se ejecuta esta celda, Spark descarga el conector y sus dependencias desde Maven Central — puede tardar. `spark.sql.shuffle.partitions` se baja a 3 (pensado para las 3 particiones originales del topic, S7 — si tu topic quedo con menos particiones, como una sola, no pasa nada, Spark igual funciona): con el valor por defecto (200), cada micro-lote crearia 200 tareas para casi ningun dato.

Verifica que el topic tiene mensajes, con una lectura **batch** (no streaming) — el mismo tipo de lectura que ya usaste con CSV o Parquet, aplicada a Kafka:

In [2]:
lote = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "atmos-eventos")
    .option("startingOffsets", "earliest")
    .load()
)
print("mensajes en el topic:", lote.count())
lote.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "partition", "offset").orderBy("offset", ascending=False).show(3, truncate=False)


mensajes en el topic: 1152


[Stage 3:>                                                          (0 + 1) / 1]

+-----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|key              |value                                                                                                                                                                        |partition|offset|
+-----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|esp32-laboratorio|{"tipoEvento": "sensor.lectura", "sensorId": "esp32-laboratorio", "temperatura": 35.9, "humedad": 31.1, "presion": 1004.4, "origen": "uso-atmos", "timestamp": 1791083554036}|0        |1154  |
|esp32-invernadero|{"tipoEvento": "sensor.lectura", "sensorId": "esp32-invernadero", "temperatura": 20.3, "humedad": 59.3, "presion": 1002.1, "origen": "uso

In [3]:
lote.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
lote.selectExpr("CAST(value AS STRING) AS mensaje").show(3, truncate=False)


+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|mensaje                                                                                                                                                                      |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"tipoEvento": "sensor.lectura", "sensorId": "esp32-patio", "temperatura": 30.6, "humedad": 30.3, "presion": 1024.5, "origen": "uso-atmos", "timestamp": 1791080859255}      |
|{"tipoEvento": "sensor.lectura", "sensorId": "esp32-invernadero", "temperatura": 29.2, "humedad": 87.5, "presion": 1000.4, "origen": "uso-atmos", "timestamp": 1791080859458}|
|{"tipoEvento": "sensor.lectura", "sensorId": "esp32-laboratorio", "temperatura": 11.1, "humedad": 35.3, "presion": 1005

In [5]:
lote.selectExpr("partition", "offset").show(3, truncate=False)


+---------+------+
|partition|offset|
+---------+------+
|0        |0     |
|0        |1     |
|0        |2     |
+---------+------+
only showing top 3 rows


Resultado esperado: un numero mayor que cero, y en las ultimas filas, JSON con `sensorId`, `temperatura`, `humedad`, `presion` (el contrato de S7). Una lectura **batch** de Kafka lee lo que hay **hoy** y termina; el resto del notebook usa lectura **streaming**, que no termina — sigue escuchando.


## 3.3 Leer el topic en modo streaming

**Producto del paso:** un DataFrame que **no se puede materializar con `.show()` ni `.count()`** — la primera diferencia real entre batch y streaming.

In [6]:
crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "atmos-eventos")
    .option("startingOffsets", "latest")
    .load()
)
print("es un stream:", crudo.isStreaming)
crudo.printSchema()


es un stream: True
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



`readStream` en vez de `read` es la unica diferencia de sintaxis, y cambia todo lo que viene despues: `crudo` es un **plan**, no una tabla en memoria. `crudo.show()` fallaria con `AnalysisException`, porque mostrar implica materializar de una vez, y un stream no tiene un "final" que mostrar. Cada mensaje de Kafka llega con las mismas columnas, sin importar el topic: `key`, `value` (ambos en binario), `topic`, `partition`, `offset`, `timestamp`. El contenido real del evento esta **adentro** de `value`, sin parsear todavia — eso es 3.4.

`startingOffsets: "latest"` (no `"earliest"`) es a proposito: en un stream que corre indefinidamente, no tiene sentido reprocesar todo el historial cada vez que se reinicia una consulta de prueba — se quiere ver lo que llega **de ahora en adelante**. La diferencia importa, y se vuelve a ver en 3.11.

## 3.4 Parsear el JSON con un esquema explicito

**Producto del paso:** las columnas reales del evento (`sensorId`, `temperatura`...), ya tipadas, listas para agregar.

In [7]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("temperatura", DoubleType())
    .add("humedad", DoubleType())
    .add("presion", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

eventos = (
    crudo.selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), esquema_evento).alias("e"))
    .select("e.*")
    .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))
)
eventos.printSchema()


root
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)



`value` llega en binario; `CAST(value AS STRING)` lo vuelve texto, y `from_json` lo convierte a columnas segun `esquema_evento` — el mismo contrato de `sensor.lectura` que S7 documento. Un esquema **explicito** (no inferido) es obligatorio en streaming: Spark no puede "mirar" todo el stream para adivinar tipos, porque el stream no tiene fin. `ts` convierte el `timestamp` (milisegundos *epoch*, un `Long`) a un tipo `timestamp` real: **es la columna que 3.5 en adelante usa como reloj del evento** — el momento en que el sensor midio, no el momento en que Spark lo procesa.

## 3.5 Primera consulta: ver los eventos llegar

**Producto del paso:** la primera consulta de streaming corriendo, con salida por consola.

In [ ]:
# Momento 1: arrancar la consulta
consulta = (
    eventos.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)

`writeStream` (no `write`) inicia una **consulta continua**: `.start()` la lanza en segundo plano y devuelve de inmediato un objeto `StreamingQuery`; el proceso sigue corriendo aunque la celda ya haya terminado — en un sistema real, una consulta de streaming no se detiene sola, corre para siempre.

`outputMode("append")` dice que solo se muestran filas **nuevas**, nunca modificadas — es el unico modo valido para eventos sin agregar, como estos. `trigger(processingTime="5 seconds")` fija cada cuanto se arma un micro-lote: Structured Streaming no procesa evento por evento, sino en **micro-lotes** cada 5 segundos. Resultado esperado: varias tandas de filas, una por cada micro-lote, con los mismos campos que en 3.1.

Dejala corriendo — **no la detengas todavia**. En 3.6 vas a arrancar una segunda consulta que lee el mismo `eventos` y lo persiste en Parquet, al mismo tiempo que esta sigue imprimiendo por consola. Las vas a detener juntas al final de 3.6.

## 3.6 Escribir la salida a Parquet, con checkpoint

**Producto del paso:** los eventos del stream aterrizando en disco, en Parquet, listos para que S9-S10 los lean como datos ya guardados.

**Opcional — filtrar antes de persistir:** `eventos` ya sale limpio de 3.4 (esquema explicito, `producer_sensores.py` siempre bien formado), asi que no hace falta para este ejercicio. Pero si la fuente fuera menos confiable, podrias descartar filas incompletas antes de guardarlas, con el mismo patron que usa `lambdalab`:

```python
eventos_validados = eventos.filter(
    col("tipoEvento").isNotNull() &
    col("sensorId").isNotNull() &
    col("temperatura").isNotNull() &
    col("ts").isNotNull()
)
```

Con datos reales de `uso-atmos` esto no cambia ni una fila (nunca llega nada nulo), asi que el resto de este notebook sigue usando `eventos` tal cual, sin esta rama adicional.

In [12]:
# Momento 1: arrancar el stream a Parquet (la consulta de 3.5 sigue corriendo en paralelo)
ARTIFACTS = "./artifacts/atmos_parquet"
CHECKPOINT_PARQUET = "./artifacts/chk_parquet"

consulta2 = (
    eventos.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", ARTIFACTS)
    .option("checkpointLocation", CHECKPOINT_PARQUET)
    .trigger(processingTime="5 seconds")
    .start()
)

-------------------------------------------
Batch: 2
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-patio      |29.7       |33.7   |1008.6 |uso-atmos|1791083961702|2026-10-04 03:19:21.702|
|sensor.lectura|esp32-invernadero|19.5       |62.8   |1000.6 |uso-atmos|1791083961705|2026-10-04 03:19:21.705|
|sensor.lectura|esp32-laboratorio|33.7       |24.9   |1009.9 |uso-atmos|1791083961709|2026-10-04 03:19:21.709|
|sensor.lectura|esp32-patio      |29.5       |33.7   |1008.7 |uso-atmos|1791083964712|2026-10-04 03:19:24.712|
|sensor.lectura|esp32-invernadero|19.6       |62.8   |1000.8 |uso-atmos|1791083964715|2026-10-04 03:19:24.715|
|sensor.lectura

Nota el nombre `consulta2`, no `consulta`: la consulta de 3.5 sigue viva en la variable `consulta`, y si esta nueva reusara ese mismo nombre, perderias la unica referencia que te permite detenerla mas tarde — quedaria corriendo huerfana, sin que ninguna variable la senale.

Mientras corre, podes leer lo que ya escribio sin detenerla — cada micro-lote confirmado queda disponible de inmediato. Este paso es **opcional**: no hace falta para completar el ejercicio, solo sirve para confirmar que realmente se esta guardando algo antes de llegar al final:

In [ ]:
# Momento 2 (opcional): leer lo guardado (se puede repetir, el stream sigue corriendo)
guardado = spark.read.parquet(ARTIFACTS)
total = guardado.count()
print("filas guardadas en Parquet:", total)
guardado.select("sensorId", "temperatura", "ts").show(total, truncate=False)

Vuelve a correr esta misma celda un par de veces, dejando pasar unos segundos entre una y otra: el conteo debe crecer cada vez, porque el stream sigue escribiendo micro-lotes nuevos mientras tanto. Cuando ya viste suficiente, detén las dos consultas juntas — la de consola de 3.5 y la de Parquet de este paso:

In [14]:
# Momento 3: detener ambas consultas (la de consola de 3.5 y la de Parquet de 3.6)
consulta.stop()
consulta2.stop()

Un *sink* de archivo (Parquet, aquí) solo acepta `outputMode("append")` — no puede "reescribir" una fila ya guardada en disco, a diferencia de `console` con `update`. Por eso esta celda escribe los eventos **sin agregar** (3.4), igual que la consulta de consola de 3.5 — no las ventanas con agregación que arrancan en 3.7: una agregación en modo `update` no se puede volcar directo a Parquet, se profundiza en 3.8-3.9.

**Error frecuente**: si lees muy rápido, antes de que llegue el primer micro-lote, el resultado es **cero filas guardadas** — no por un error, sino porque no hubo datos nuevos todavía. Espera unos segundos y vuelve a correr la celda de lectura; si sigue en 0, confirma que el productor de 3.1 sigue corriendo.

## 3.7 Agregar por ventana de tiempo, sin limite (el problema que resuelve 3.8)

**Producto del paso:** evidencia de que, sin watermark, el estado de una agregacion por ventana **crece para siempre**.

In [ ]:
# Momento 1: arrancar la agregacion sin limite
from pyspark.sql.functions import window, avg, count

agregado_sin_limite = (
    eventos.groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(avg("temperatura").alias("temp_prom"), count("*").alias("n"))
)

consulta = (
    agregado_sin_limite.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)

Dejala correr al menos 20-25 segundos (varios micro-lotes) para que se note la tabla creciendo, y despues detenla:

In [16]:
# Momento 2: detener la consulta
consulta.stop()

`window(col("ts"), "10 seconds")` agrupa los eventos en bloques de 10 segundos de tiempo de **evento** (`ts`), no de tiempo de procesamiento — dos sensores que midieron en el mismo instante caen en la misma ventana, sin importar cuando Spark los proceso. `outputMode("complete")` es el unico modo que puede acompanar a una agregacion **sin** watermark: en cada micro-lote, Spark reimprime **todas** las ventanas vistas hasta ahora, porque no tiene ninguna senal de que una ventana vieja ya no va a recibir mas datos.

Observa que, de un micro-lote al siguiente, la tabla completa **crece**: las ventanas de hace 20 segundos siguen apareciendo, iguales, junto a las nuevas. Ese es el problema: Spark tiene que guardar el estado de **cada ventana que existio jamas**, indefinidamente — en un sistema real, que corre por dias, esto agota la memoria. El watermarking (3.8) es la respuesta.

## 3.8 Watermarking: acotar cuanto se espera por datos tardios

**Producto del paso:** la misma agregacion, pero con las ventanas viejas **cerradas y liberadas** de la memoria.

In [ ]:
# Momento 1: arrancar la agregacion con watermark
agregado_con_watermark = (
    eventos
    .withWatermark("ts", "10 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(avg("temperatura").alias("temp_prom"), count("*").alias("n"))
)

consulta = (
    agregado_con_watermark.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-watermark")
    .trigger(processingTime="5 seconds")
    .start()
)

Dejala correr al menos 20-25 segundos y detenla:

In [18]:
# Momento 2: detener la consulta
consulta.stop()

`withWatermark("ts", "10 seconds")` es la regla que le faltaba a 3.7: le dice a Spark "una vez que veas un evento con `ts` = X, ya no esperes datos con `ts` anterior a X menos 10 segundos — cierra esas ventanas y olvidalas". El **watermark** en un momento dado es, entonces, el evento mas reciente visto menos ese margen. Con el watermark, `outputMode("update")` ya es valido: en cada micro-lote solo se imprimen las ventanas que **cambiaron**, no todas — y las ventanas cerradas dejan de aparecer y dejan de ocupar memoria.

`checkpointLocation` aparece por primera vez: es la carpeta donde Spark guarda el estado de la consulta (que ventanas existen, hasta que offset de Kafka se leyo). Sin ella, esta consulta con estado (una agregacion) ni siquiera arrancaria. Se profundiza en 3.10.

Compara la salida con la de 3.7: aca la tabla **no** repite ventanas viejas en cada lote — cada micro-lote solo trae lo nuevo o lo que cambio.

## 3.9 Ventana deslizante: una lectura que se actualiza mas seguido que se cierra

**Producto del paso:** ventanas que se **solapan**, para suavizar una metrica sin esperar a que cada bloque termine.

In [ ]:
# Momento 1: arrancar la ventana deslizante
agregado_deslizante = (
    eventos
    .withWatermark("ts", "15 seconds")
    .groupBy(window(col("ts"), "15 seconds", "5 seconds"), col("sensorId"))
    .agg(avg("temperatura").alias("temp_prom"), count("*").alias("n"))
)

consulta = (
    agregado_deslizante.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-deslizante")
    .trigger(processingTime="5 seconds")
    .start()
)

Dejala correr al menos 20-25 segundos y detenla:

In [20]:
# Momento 2: detener la consulta
consulta.stop()

`window(col("ts"), "15 seconds", "5 seconds")` tiene un tercer argumento que 3.8 no tenia: el **paso** (5 segundos) es menor que el **ancho** (15 segundos) de la ventana. El resultado es una ventana **deslizante**: cada evento cae en **varias** ventanas de 15 segundos a la vez (una que empieza ahora, otra que empezo hace 5 segundos, otra hace 10). En 3.8, cada evento caia en **una sola** ventana (ventana **fija** o *tumbling*: el paso es igual al ancho).

Observa en la salida que, para un mismo `sensorId`, aparecen varias filas con rangos de `window` que se superponen — y el mismo evento cuenta para varias de ellas. Sirve para suavizar una lectura (un promedio que no salta de golpe cada 15 segundos), a costa de guardar mas estado: cada evento se cuenta varias veces en la memoria de la consulta, no una.

## 3.10 Checkpointing: que guarda, y que pasa al reiniciar

**Producto del paso:** evidencia de que el *checkpoint* no es una carpeta vacia — ahi vive el progreso real de la consulta.

In [21]:
import os

for carpeta in sorted(os.listdir("/tmp/s08-chk-watermark")):
    print(carpeta)


.metadata.crc
commits
metadata
offsets
sources
state


In [22]:
archivos_offsets = sorted(os.listdir("/tmp/s08-chk-watermark/offsets"))
ultimo_offset = archivos_offsets[-1]
with open(f"/tmp/s08-chk-watermark/offsets/{ultimo_offset}") as f:
    print(f.read())


v1
{"batchWatermarkMs":1791084138458,"batchTimestampMs":1791084155008,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.stateStore.rocksdb.mergeOperatorVersion":"2","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.queryEvolution.enableSourceEvolution":"false","spark.sql.streaming.stateStore.encodingFormat":"unsaferow","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.stateStore.rowChecksum.enabled":"false","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.function.protobufExtensions.enabled":"false","spark.sql.shuffle.partitions":"3","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.optimizer.pruneFiltersCanPrune

`offsets/` guarda, por cada micro-lote, hasta que *offset* de cada particion de Kafka se leyo — en esta corrida, con el topic de una sola particion, la particion 0 llego hasta el mensaje 1752 (si tu topic tiene varias particiones, vas a ver una clave por cada una). `commits/` confirma cuales de esos micro-lotes terminaron de escribirse por completo. `state/` guarda el contenido de cada ventana abierta — es lo que 3.7 mostro creciendo sin limite.

La consecuencia practica: si una consulta se cae y se reinicia **apuntando al mismo `checkpointLocation`**, Spark lee `offsets/` y retoma exactamente donde se quedo — no vuelve a leer el topic desde el principio, y no pierde el estado de las ventanas que ya tenia abiertas. Es la misma idea del *offset* de un `consumer group` de Kafka (S6, S7), pero aplicada tambien al estado de la agregacion, no solo a la posicion de lectura.

## 3.11 Un dato tardio, y el watermark descartandolo

**Producto del paso:** un evento con marca de tiempo vieja, publicado **despues** de que el watermark ya avanzo mas alla de su ventana — y la prueba de que Spark lo descarta sin avisar.

**Advertencia sobre el "arranque en frio":** si publicas el evento tardio como el **primer** mensaje que la consulta ve, no se descarta — el watermark todavia no tiene ningun evento "reciente" contra el cual compararlo. Por eso, primero se deja correr la consulta con datos en vivo un rato, y **recien despues** se publica el evento tardio.

In [ ]:
# Momento 1: arrancar el hilo y la consulta
import threading
import time
import json
from pyspark.sql.functions import to_json, struct

def publicar_evento_tardio(segundos_de_espera, segundos_en_el_pasado):
    time.sleep(segundos_de_espera)
    ahora_ms = int(time.time() * 1000)
    evento_tardio = [(
        "sensor.lectura", "esp32-tardio", 99.9, 50.0, 1000.0,
        "prueba-watermark", ahora_ms - segundos_en_el_pasado * 1000
    )]
    columnas = ["tipoEvento", "sensorId", "temperatura", "humedad", "presion", "origen", "timestamp"]
    df_tardio = spark.createDataFrame(evento_tardio, columnas)
    salida = df_tardio.select(
        df_tardio.sensorId.cast("string").alias("key"),
        to_json(struct(*columnas)).alias("value"),
    )
    (salida.write.format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("topic", "atmos-eventos")
        .save())
    print(f">>> publicado esp32-tardio, {segundos_en_el_pasado}s en el pasado")

hilo = threading.Thread(target=publicar_evento_tardio, args=(15, 45), daemon=True)
hilo.start()

agregado = (
    eventos
    .withWatermark("ts", "5 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(count("*").alias("n"))
)

consulta = (
    agregado.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-tardio")
    .trigger(processingTime="4 seconds")
    .start()
)

El hilo (`daemon=True`) corre aparte de la consulta: se dispara solo, 15 segundos despues de ejecutar esta celda, sin importar cuando decidas detener `consulta`. Espera **al menos 30-40 segundos** antes de detenerla — necesitas ver varios micro-lotes con datos en vivo antes del evento tardio, y varios mas despues, para comprobar que nunca aparece:

In [24]:
# Momento 2: detener la consulta y revisar
consulta.stop()
print("Buscando 'esp32-tardio' en la salida de arriba: si no aparece ninguna fila, el watermark lo descarto.")

Buscando 'esp32-tardio' en la salida de arriba: si no aparece ninguna fila, el watermark lo descarto.


El hilo publica un evento `esp32-tardio` cuyo `timestamp` corresponde a **45 segundos antes** del momento de publicacion, pero recien 15 segundos despues de arrancar la consulta — tiempo suficiente para que el watermark ya haya avanzado con los eventos en vivo del productor de S7. Con un watermark de 5 segundos, cualquier ventana que termine mas de 5 segundos antes del evento mas reciente visto queda cerrada, y un evento que llega despues para esa ventana se descarta **en silencio**: no hay excepcion, no hay log de error, la fila simplemente nunca aparece en la salida.

Esto se verifico con datos reales antes de escribir este notebook: el mismo experimento, publicando el evento tardio **antes** de que el watermark avanzara (como primer mensaje de la consulta), si lo dejaba pasar — es el caso de "arranque en frio" de la advertencia de arriba. La leccion no es "Spark falla a veces", es que el watermark necesita **datos frescos previos** para tener algo contra qué comparar.


## 3.12 Deduplicacion acotada por watermark

**Producto del paso:** el mismo evento, publicado dos veces por error, contado **una sola vez**.

In [ ]:
# Momento 1: arrancar la consulta
deduplicado = (
    eventos
    .withWatermark("ts", "30 seconds")
    .dropDuplicates(["sensorId", "timestamp"])
)

consulta = (
    deduplicado.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-dedup")
    .trigger(processingTime="3 seconds")
    .start()
)

Con la consulta corriendo, espera unos segundos a que arranque y publica el mismo evento dos veces seguidas:

In [ ]:
# Momento 2: publicar el evento duplicado
time.sleep(6)
columnas = ["tipoEvento", "sensorId", "temperatura", "humedad", "presion", "origen", "timestamp"]
evento_duplicado = spark.createDataFrame(
    [("sensor.lectura", "esp32-duplicado", 55.0, 55.0, 1000.0, "prueba-dedup", int(time.time() * 1000))],
    columnas,
)
salida = evento_duplicado.select(
    evento_duplicado.sensorId.cast("string").alias("key"),
    to_json(struct(*columnas)).alias("value"),
)
for intento in range(2):
    (salida.write.format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("topic", "atmos-eventos")
        .save())
print(">>> 'esp32-duplicado' publicado DOS veces")

Espera unos segundos mas a que pasen un par de micro-lotes mostrando el filtrado, y detenla:

In [27]:
# Momento 3: detener y revisar
consulta.stop()
print("Busca 'esp32-duplicado' arriba: debe aparecer una sola vez, aunque se publico dos veces.")

Busca 'esp32-duplicado' arriba: debe aparecer una sola vez, aunque se publico dos veces.


`dropDuplicates(["sensorId", "timestamp"])` descarta cualquier evento cuya combinacion de esas dos columnas ya se vio antes. Sin `withWatermark`, esta operacion tendria que recordar **todos** los eventos vistos jamas, para siempre — el mismo problema sin limite de 3.7, pero sobre los propios datos en vez de sobre ventanas. El watermark acota cuanto se recuerda: pasado el margen (30 segundos), un identificador viejo se olvida, y un duplicado *muy* tardio ya no se detectaria — es el mismo balance de siempre entre memoria y exactitud.

Esto responde directamente al porque de **semantica de entrega**: Kafka entrega cada mensaje **al menos una vez** (S6, S7) — nunca menos, a veces mas, por reintentos ante fallos de red. `dropDuplicates` acotado por watermark es la pieza que convierte ese "al menos una vez" en un resultado que se comporta como "exactamente una vez", **sin** que el productor tenga que garantizar nada.

## 3.13 Intervalo de disparo: el costo de decidir cada cuanto procesar

**Producto del paso:** el mismo stream, con dos ritmos de disparo distintos, y la diferencia real en cuantos micro-lotes se alcanzan a correr.

A diferencia de las consultas anteriores, esta celda si usa `awaitTermination(segundos_totales)` en vez de un `stop()` manual aparte: la comparacion solo es justa si ambos disparos corren exactamente el mismo tiempo de reloj (15 segundos cada uno) — si los detuvieras a mano, nunca sabrias si la diferencia de `batchId` vino del intervalo o de cuanto esperaste.

In [ ]:
def contar_microlotes(intervalo, segundos_totales):
    consulta = (
        eventos.writeStream
        .outputMode("append")
        .format("memory")
        .queryName(f"conteo_{intervalo.replace(' ', '_')}")
        .trigger(processingTime=intervalo)
        .start()
    )
    consulta.awaitTermination(segundos_totales)
    consulta.stop()
    return consulta.lastProgress["batchId"] if consulta.lastProgress else None

ultimo_lote_rapido = contar_microlotes("1 second", 15)
ultimo_lote_lento = contar_microlotes("6 seconds", 15)

print("Con disparo cada 1 segundo, ultimo batchId en 15s:", ultimo_lote_rapido)
print("Con disparo cada 6 segundos, ultimo batchId en 15s:", ultimo_lote_lento)


`format("memory")` guarda la salida en una tabla temporal en vez de imprimirla — util para inspeccionar resultados con SQL, y aca para medir sin llenar la pantalla de filas. `consulta.lastProgress["batchId"]` es el numero del ultimo micro-lote que corrio: con el disparo de 1 segundo, en 15 segundos de reloj corren muchos mas micro-lotes que con el de 6 segundos.

La diferencia es el compromiso entre **latencia** y **throughput** (rendimiento) que pide el silabo: un disparo mas frecuente reduce la latencia (un evento espera menos, en promedio, antes de procesarse) pero cada micro-lote es mas chico y el overhead de coordinarlos se paga mas veces por segundo; un disparo menos frecuente junta mas eventos por lote (mejor throughput por lote) a cambio de que cada evento individual espere mas antes de aparecer en el resultado. Ninguno es "el correcto": depende de si el sistema necesita reaccionar rapido (alertas) o procesar volumen (reportes).


**Error frecuente**: el log puede mostrar lineas en rojo `ERROR WriteToDataSourceV2Exec`, `TaskKilledException` o `Aborting task` — igual que en 3.5, es ruido esperado del `consulta.stop()` dentro de `contar_microlotes()`, que corta la consulta a mitad de un micro-lote. No es una falla: mientras la celda termine imprimiendo los dos `print()` finales (sin lanzar una excepcion de Python), los resultados son correctos.

## 3.14 Documentar hallazgos y responder preguntas de reflexion

**Producto del paso:** una celda markdown que dice, en tus propias palabras, que efecto tuvo cada parametro — sin volver a leer el notebook para recordarlo.

Responde, con los resultados reales que obtuviste arriba:

1. En 3.7, ¿que le pasaba a la tabla impresa en cada micro-lote, y por que dejo de pasar en 3.8?
2. En 3.11, ¿el evento tardio aparecio o no en la salida? ¿Cuantos segundos de diferencia habia entre su `timestamp` y el watermark del momento en que se publico?
3. En 3.12, ¿cuantas veces se publico `esp32-duplicado`, y cuantas veces aparecio en la salida deduplicada?
4. En 3.13, ¿que `batchId` alcanzo cada disparo (1 segundo vs. 6 segundos) en los mismos 15 segundos de reloj? ¿Cual eligirias para un panel de alertas en vivo, y cual para un reporte nocturno?

**(Completa aca, con tus propios resultados)**

1.
2.
3.
4.
